# Transformer-Based NMT

### Preparation

In [22]:
import json
import math
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass

try:
    import jieba
except ImportError:
    jieba = None

try:
    import sacrebleu
except ImportError:
    sacrebleu = None


seed = 250010155
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

### Tokenization & Vocab

In [23]:
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = SPECIALS

def tokenize_zh(s: str) -> List[str]:
    s = s.strip()
    if jieba is None:
        # fallback: char-level
        return list(s)
    return list(jieba.cut(s, cut_all=False))

def tokenize_en(s: str) -> List[str]:
    # simple whitespace tokenization
    return s.strip().split()

class Vocab:
    def __init__(self, min_freq: int = 2, max_size: int = 50000):
        self.min_freq = min_freq
        self.max_size = max_size
        self.freq: Dict[str, int] = {}
        self.stoi: Dict[str, int] = {}
        self.itos: List[str] = []

    def add_sentence(self, tokens: List[str]):
        for t in tokens:
            self.freq[t] = self.freq.get(t, 0) + 1

    def build(self):
        items = [(t, c) for t, c in self.freq.items() if c >= self.min_freq]
        items.sort(key=lambda x: (-x[1], x[0]))
        items = items[: max(0, self.max_size - len(SPECIALS))]

        self.itos = SPECIALS + [t for t, _ in items]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def encode(self, tokens: List[str]) -> List[int]:
        return [self.stoi.get(t, self.stoi[UNK]) for t in tokens]

    def decode(self, ids: List[int], stop_at_eos: bool = True) -> List[str]:
        out = []
        for i in ids:
            if i < 0 or i >= len(self.itos):
                tok = UNK
            else:
                tok = self.itos[i]
            if stop_at_eos and tok == EOS:
                break
            out.append(tok)
        return out

    @property
    def pad_id(self) -> int: return self.stoi[PAD]
    @property
    def bos_id(self) -> int: return self.stoi[BOS]
    @property
    def eos_id(self) -> int: return self.stoi[EOS]
    @property
    def unk_id(self) -> int: return self.stoi[UNK]
    def __len__(self): return len(self.itos)

### Dataset

In [24]:
class JsonlParallelDataset(Dataset):
    def __init__(
        self,
        path: str,
        src_key: str,
        tgt_key: str,
        src_tok,
        tgt_tok,
        src_vocab: Optional[Vocab] = None,
        tgt_vocab: Optional[Vocab] = None,
        max_len: int = 120,
        build_vocab: bool = False,
    ):
        self.data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                src = obj[src_key]
                tgt = obj[tgt_key]
                src_tokens = src_tok(src)
                tgt_tokens = tgt_tok(tgt)
                if len(src_tokens) == 0 or len(tgt_tokens) == 0:
                    continue
                if len(src_tokens) > max_len or len(tgt_tokens) > max_len:
                    continue
                self.data.append((src_tokens, tgt_tokens))

        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

        if build_vocab:
            assert src_vocab is not None and tgt_vocab is not None
            for s, t in self.data:
                src_vocab.add_sentence(s)
                tgt_vocab.add_sentence(t)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch, src_vocab: Vocab, tgt_vocab: Vocab, device: torch.device):
    src_ids = []
    tgt_ids = []
    for src_tok, tgt_tok in batch:
        s = [src_vocab.bos_id] + src_vocab.encode(src_tok) + [src_vocab.eos_id]
        t = [tgt_vocab.bos_id] + tgt_vocab.encode(tgt_tok) + [tgt_vocab.eos_id]
        src_ids.append(torch.tensor(s, dtype=torch.long))
        tgt_ids.append(torch.tensor(t, dtype=torch.long))

    src_lens = torch.tensor([len(x) for x in src_ids], dtype=torch.long)
    tgt_lens = torch.tensor([len(x) for x in tgt_ids], dtype=torch.long)

    src_pad = pad_sequence(src_ids, batch_first=True, padding_value=src_vocab.pad_id).to(device)
    tgt_pad = pad_sequence(tgt_ids, batch_first=True, padding_value=tgt_vocab.pad_id).to(device)

    return src_pad, src_lens.to(device), tgt_pad, tgt_lens.to(device)

### Positional Encoding

In [25]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)  # [L, D]
        pos = torch.arange(0, max_len).unsqueeze(1).float()  # [L,1]
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)  # [L,D]

    def forward(self, x: torch.Tensor):
        # x: [B,T,D] (batch_first)
        T = x.size(1)
        x = x + self.pe[:T].unsqueeze(0)
        return self.dropout(x)

def generate_square_subsequent_mask(sz: int, device: torch.device):
    # True means "masked" in PyTorch Transformer if using float mask it's -inf
    # We'll return float mask: [T,T] with -inf above diagonal
    mask = torch.full((sz, sz), float("-inf"), device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask


### Transformer

In [39]:
class TransformerNMT(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        pad_id: int,
        d_model: int = 256,
        nhead: int = 4,
        num_encoder_layers: int = 4,
        num_decoder_layers: int = 4,
        dim_feedforward: int = 1024,
        dropout: float = 0.1,
        max_len: int = 5000,
        pos_type: str = 'sin'
    ):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)

        self.pos_type = pos_type
        if self.pos_type == "learned":
            self.pos_emb = nn.Embedding(max_len, d_model)
        else:
            self.pos = PositionalEncoding(d_model, dropout=dropout, max_len=max_len)


        self.tf = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # IMPORTANT
            norm_first=True,    # tends to be a bit more stable
        )
        self.proj = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src_ids: torch.Tensor, tgt_inp_ids: torch.Tensor):
        """
        src_ids: [B,S]
        tgt_inp_ids: [B,T]  (typically tgt[:, :-1])
        return logits: [B,T,V]
        """
        device = src_ids.device
        B, S = src_ids.shape
        B2, T = tgt_inp_ids.shape
        assert B == B2

        # padding masks: True means "ignore" for nn.Transformer
        src_key_padding_mask = (src_ids == self.pad_id)      # [B,S]
        tgt_key_padding_mask = (tgt_inp_ids == self.pad_id)  # [B,T]

        # causal mask for decoder self-attention
        tgt_mask = generate_square_subsequent_mask(T, device=device)  # [T,T] float(-inf) upper

        src = self.pos(self.src_emb(src_ids) * math.sqrt(self.d_model))  # [B,S,D]
        tgt = self.pos(self.tgt_emb(tgt_inp_ids) * math.sqrt(self.d_model))  # [B,T,D]

        if self.pos_type == "learned":
            # positions [0..S-1], [0..T-1]
            S = src_ids.size(1); T = tgt_inp_ids.size(1)
            pos_s = torch.arange(S, device=src_ids.device).unsqueeze(0)
            pos_t = torch.arange(T, device=src_ids.device).unsqueeze(0)
            src = src + self.pos_emb(pos_s)
            tgt = tgt + self.pos_emb(pos_t)
            src = F.dropout(src, p=self.tf.dropout, training=self.training)
            tgt = F.dropout(tgt, p=self.tf.dropout, training=self.training)
        else:
            src = self.pos(src)
            tgt = self.pos(tgt)

        out = self.tf(
            src=src,
            tgt=tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )  # [B,T,D]
        logits = self.proj(out)  # [B,T,V]
        return logits

    @torch.no_grad()
    def greedy(self, src_ids: torch.Tensor, bos_id: int, eos_id: int, max_len: int = 120):
        """
        src_ids: [B,S]
        return list[list[int]] of generated ids (without BOS)
        """
        self.eval()
        device = src_ids.device
        B = src_ids.size(0)

        ys = torch.full((B, 1), bos_id, dtype=torch.long, device=device)  # [B,1]
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_len):
            logits = self.forward(src_ids, ys)  # [B,t,V]
            next_tok = torch.argmax(logits[:, -1, :], dim=1)  # [B]
            ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)

            finished = finished | (next_tok == eos_id)
            if torch.all(finished):
                break

        # strip BOS and stop at EOS
        outs = []
        for b in range(B):
            seq = ys[b, 1:].tolist()
            if eos_id in seq:
                seq = seq[: seq.index(eos_id)]
            outs.append(seq)
        return outs


    @torch.no_grad()
    def beam(
        self,
        src_ids: torch.Tensor,
        bos_id: int,
        eos_id: int,
        beam_size: int = 5,
        max_len: int = 120,
        len_norm_alpha: float = 0.6,
    ):
        """
        Beam search decoding.
        For robustness, this implementation assumes batch_size=1.
        Returns: list[int] (without BOS, stop at EOS)
        """
        self.eval()
        assert src_ids.size(0) == 1, "beam() here assumes batch_size=1."
        device = src_ids.device

        beams = [([bos_id], 0.0, False)]  # (tokens, logp, ended)

        for _ in range(max_len):
            new_beams = []
            all_ended = True

            for tokens, lp, ended in beams:
                if ended:
                    new_beams.append((tokens, lp, True))
                    continue

                all_ended = False
                ys = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)  # [1,t]
                logits = self.forward(src_ids, ys)  # [1,t,V]
                logp = F.log_softmax(logits[0, -1], dim=-1)  # [V]

                # 防止第一步直接 EOS -> 空句（和你 RNN 的修复同理）
                if len(tokens) == 1:
                    logp[eos_id] = float("-inf")

                topk = torch.topk(logp, k=beam_size)
                for k in range(beam_size):
                    tok = int(topk.indices[k].item())
                    score = float(topk.values[k].item())
                    nt = tokens + [tok]
                    nlp = lp + score
                    nended = (tok == eos_id)
                    new_beams.append((nt, nlp, nended))

            if all_ended:
                break

            # length normalization (Google NMT style)
            def norm(item):
                tokens, lp, _ = item
                L = max(1, len(tokens) - 1)  # exclude BOS
                return lp / (((5 + L) ** len_norm_alpha) / (5 ** len_norm_alpha))

            new_beams.sort(key=norm, reverse=True)
            beams = new_beams[:beam_size]

        best_tokens = max(beams, key=lambda x: x[1])[0]  # raw logp best
        out = best_tokens[1:]  # remove BOS
        if eos_id in out:
            out = out[: out.index(eos_id)]
        return out

### Loss

In [35]:
def xent_loss(logits: torch.Tensor, tgt_gold: torch.Tensor, pad_id: int, label_smooth: float = 0.0):
    """
    logits: [B,T,V]
    tgt_gold: [B,T]
    """
    B, T, V = logits.shape
    logits = logits.reshape(B*T, V)
    tgt = tgt_gold.reshape(B*T)

    if label_smooth <= 0.0:
        return F.cross_entropy(logits, tgt, ignore_index=pad_id)

    # label smoothing
    logp = F.log_softmax(logits, dim=-1)
    nll = F.nll_loss(logp, tgt, ignore_index=pad_id, reduction="none")
    smooth = -logp.mean(dim=-1)

    mask = (tgt != pad_id)
    nll = nll[mask]
    smooth = smooth[mask]
    return (1.0 - label_smooth) * nll.mean() + label_smooth * smooth.mean()

### Train & Eval

In [36]:
def train_one_epoch_transformer(model: TransformerNMT, loader, optim, pad_id: int, label_smooth: float, clip: float):
    model.train()
    total, n = 0.0, 0

    for src_ids, src_lens, tgt_ids, tgt_lens in tqdm(loader, desc="train", leave=False):
        # teacher forcing: input is tgt[:, :-1], gold is tgt[:, 1:]
        tgt_inp = tgt_ids[:, :-1]
        tgt_gold = tgt_ids[:, 1:]

        optim.zero_grad()
        logits = model(src_ids, tgt_inp)  # [B,T-1,V]
        loss = xent_loss(logits, tgt_gold, pad_id=pad_id, label_smooth=label_smooth)
        loss.backward()
        clip_grad_norm_(model.parameters(), clip)
        optim.step()

        total += float(loss.item()); n += 1

    return total / max(1, n)

@torch.no_grad()
def eval_bleu_transformer(model: TransformerNMT, loader, src_vocab, tgt_vocab, bos_id: int, eos_id: int, decode: str = "greedy", max_len: int = 120):
    if sacrebleu is None:
        print("[WARN] sacrebleu not installed; BLEU skipped.")
        return None

    model.eval()
    hyps, refs = [], []
    FILTER = {"<pad>", "<bos>", "<eos>"}  # 不要过滤 <unk>

    for src_ids, src_lens, tgt_ids, _ in tqdm(loader, desc="eval", leave=False):
        # refs
        for b in range(tgt_ids.size(0)):
            ref_tokens = tgt_vocab.decode(tgt_ids[b].tolist()[1:], stop_at_eos=True)
            refs.append(" ".join([t for t in ref_tokens if t not in FILTER]))

        # hyps
        if decode == "greedy":
            out_ids = model.greedy(src_ids, bos_id, eos_id, max_len=max_len)
        else:
            out_ids = []
            for b in range(src_ids.size(0)):
                hyp = model.beam(src_ids[b:b+1], bos_id, eos_id, beam_size=5, max_len=max_len)
                out_ids.append(hyp)

        for b in range(len(out_ids)):
            hyp_tokens = tgt_vocab.decode(out_ids[b], stop_at_eos=True)
            hyp = " ".join([t for t in hyp_tokens if t not in FILTER])
            hyps.append(hyp)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score



@dataclass
class TFTrainCfg:
    lr: float = 5e-4
    clip: float = 1.0
    label_smooth: float = 0.1

def run_train_transformer(args):
    device = torch.device(args.device)

    # 1) build vocab
    src_vocab = Vocab(min_freq=args.min_freq, max_size=args.vocab_size)
    tgt_vocab = Vocab(min_freq=args.min_freq, max_size=args.vocab_size)

    train_ds_for_vocab = JsonlParallelDataset(
        args.train, args.src_key, args.tgt_key,
        src_tok=tokenize_zh, tgt_tok=tokenize_en,
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        max_len=args.max_len, build_vocab=True
    )
    src_vocab.build()
    tgt_vocab.build()
    print(f"src_vocab={len(src_vocab)}, tgt_vocab={len(tgt_vocab)}, train_samples={len(train_ds_for_vocab)}")

    # 2) loaders
    train_ds = train_ds_for_vocab
    valid_ds = JsonlParallelDataset(
        args.valid, args.src_key, args.tgt_key,
        src_tok=tokenize_zh, tgt_tok=tokenize_en,
        src_vocab=src_vocab, tgt_vocab=tgt_vocab,
        max_len=args.max_len, build_vocab=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=args.batch, shuffle=True,
        collate_fn=lambda b: collate_fn(b, src_vocab, tgt_vocab, device),
        num_workers=0
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=min(args.batch, 64), shuffle=False,
        collate_fn=lambda b: collate_fn(b, src_vocab, tgt_vocab, device),
        num_workers=0
    )

    # 3) model
    model = TransformerNMT(
        src_vocab_size=len(src_vocab),
        tgt_vocab_size=len(tgt_vocab),
        pad_id=tgt_vocab.pad_id,         # target pad_id for loss and tgt padding
        d_model=args.d_model,
        nhead=args.nhead,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        dim_feedforward=args.ffn_dim,
        dropout=args.dropout,
    ).to(device)

    optim = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.98), eps=1e-9)
    cfg = TFTrainCfg(lr=args.lr, clip=args.clip, label_smooth=args.label_smooth)

    best = -1e9
    for ep in range(1, args.epochs + 1):
        tr_loss = train_one_epoch_transformer(model, train_loader, optim, pad_id=tgt_vocab.pad_id,
                                             label_smooth=cfg.label_smooth, clip=cfg.clip)
        bleu = eval_bleu_transformer(model, valid_loader, src_vocab, tgt_vocab, tgt_vocab.bos_id, tgt_vocab.eos_id,
                                     decode=args.decode, max_len=args.max_len)
        bleu_str = "N/A" if bleu is None else f"{bleu:.2f}"
        print(f"[epoch {ep}] train_loss={tr_loss:.4f} valid_BLEU={bleu_str}")

        score = -tr_loss if bleu is None else bleu
        if score > best:
            best = score
            ckpt = {
                "args": vars(args),
                "src_vocab": {"itos": src_vocab.itos, "stoi": src_vocab.stoi},
                "tgt_vocab": {"itos": tgt_vocab.itos, "stoi": tgt_vocab.stoi},
                "model": model.state_dict(),
            }
            torch.save(ckpt, args.save)
            print(f"  saved: {args.save}")

    return model, (src_vocab, tgt_vocab)

### Main

In [52]:
from types import SimpleNamespace

tf_args = SimpleNamespace(
    train="data/train_10k.jsonl",
    valid="data/valid.jsonl",
    src_key="zh",
    tgt_key="en",
    max_len=120,
    min_freq=2,
    vocab_size=50000,

    # transformer
    d_model=128,
    nhead=4,
    enc_layers=2,
    dec_layers=2,
    ffn_dim=512,
    dropout=0.1,

    batch=64,
    epochs=10,
    lr=5e-4,
    clip=1.0,
    label_smooth=0.1,
    pos_type='sin',

    decode="beam",
    beam_size=5,
    device="cuda" if torch.cuda.is_available() else "cpu",
    save="tf_nmt.pt",
)

In [53]:
tf_model, (src_vocab, tgt_vocab) = run_train_transformer(tf_args)

c:\Users\250010155\AppData\Local\miniconda3\envs\py310\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


src_vocab=9687, tgt_vocab=12485, train_samples=9998


train:   0%|          | 0/157 [00:00<?, ?it/s]c:\Users\250010155\AppData\Local\miniconda3\envs\py310\lib\site-packages\torch\nn\functional.py:5849: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


[epoch 1] train_loss=7.6723 valid_BLEU=0.00
  saved: tf_nmt.pt


[epoch 2] train_loss=6.9538 valid_BLEU=0.00


...

[epoch 6] train_loss=6.3356 valid_BLEU=17.95
  saved: tf_nmt.pt


[epoch 7] train_loss=6.2257 valid_BLEU=21.65
  saved: tf_nmt.pt


[epoch 8] train_loss=6.1272 valid_BLEU=20.00


[epoch 9] train_loss=6.0370 valid_BLEU=16.21


[epoch 10] train_loss=5.9544 valid_BLEU=17.51
